In [0]:
data=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17)]

schema = ['id','name','sal','mngr_id']
manager_df = spark.createDataFrame(data=data,schema=schema)

In [0]:
manager_df.show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| Vikas|75000|     16|
| 12| Nisha|40000|     18|
| 13| Nidhi|60000|     17|
| 14| Priya|80000|     18|
| 15| Mohit|45000|     18|
| 16|Rajesh|90000|     10|
| 17| Raman|55000|     16|
| 18|   Sam|65000|     17|
+---+------+-----+-------+



In [0]:
data1=[(19 ,'Sohan',50000, 18),
(20 ,'Sima',75000,  17)]
schema1=['id','name','sal','mngr_id']
manager_df1=spark.createDataFrame(data=data1,schema=schema1)

In [0]:
manager_df.union(manager_df1).show()

+---+------+-----+-------+
| id|  name|  sal|mngr_id|
+---+------+-----+-------+
| 10|  Anil|50000|     18|
| 11| Vikas|75000|     16|
| 12| Nisha|40000|     18|
| 13| Nidhi|60000|     17|
| 14| Priya|80000|     18|
| 15| Mohit|45000|     18|
| 16|Rajesh|90000|     10|
| 17| Raman|55000|     16|
| 18|   Sam|65000|     17|
| 19| Sohan|50000|     18|
| 20|  Sima|75000|     17|
+---+------+-----+-------+



In [0]:
manager_df.union(manager_df1).count()

11

In [0]:
manager_df.union(manager_df1).count()

11

In [0]:
manager_df.union(manager_df1).count()

11

In [0]:
data2=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17),
(18 ,'Sam',55000,   17),
(18 ,'Sam',65000,   17)]
schema=['id','name','sal','mngr_id']
duplicate_manager_df=spark.createDataFrame(data=data2,schema=schema)

In [0]:
duplicate_manager_df.union(manager_df1).count()

13

In [0]:
duplicate_manager_df.unionAll(manager_df1).count()

13

##### When we work on DataFrame both union and union All will give same result but If we work on spark sql it will throw different result.

In [0]:
manager_df1.createOrReplaceTempView("manager_df1_tbl")
duplicate_manager_df.createOrReplaceTempView("duplicate_manager_df_tbl")

##### Now we are doing Union and Union All on spark sql.
- UNION in SQL: This operator combines the result sets and automatically removes duplicate rows from the final output. It ensures that each row in the combined result set is unique. This deduplication process involves sorting the combined data, which can impact performance, especially with large datasets.<br>
- UNION ALL in SQL: This operator also combines the result sets, but it includes all rows from the individual SELECT statements, including any duplicate rows. No deduplication process is performed, making UNION ALL generally faster than UNION because it avoids the overhead of sorting and removing duplicates.

In [0]:
spark.sql("""
          Select * from manager_df1_tbl
          union
          select * from duplicate_manager_df_tbl
          """).count()

12

In [0]:
spark.sql("""
          Select * from manager_df1_tbl
          union all
          select * from duplicate_manager_df_tbl
          """).count()

13

We are creating a dataframe wrong_manager_df where column order is: <id, sal, mngr_id, Name> And another dataframe manager_df1 where column order is : <id, name, sal, mngr_id>. So, in both dataframe column is not in proper order in that case if we apply union what will happen?

In [0]:
wrong_column_data=[(19 ,50000, 18,'Sohan'),
(20 ,75000,  17,'Sima')]
Wrong_schema =['id','sal','mngr_id','Name']
Wrong_manager_df=spark.createDataFrame(data=wrong_column_data,schema=Wrong_schema)

In [0]:
manager_df1.union(Wrong_manager_df).show()

+---+-----+-----+-------+
| id| name|  sal|mngr_id|
+---+-----+-----+-------+
| 19|Sohan|50000|     18|
| 20| Sima|75000|     17|
| 19|50000|   18|  Sohan|
| 20|75000|   17|   Sima|
+---+-----+-----+-------+



So, when column will not be in order in both dataframe then we can apply unionByName :

In [0]:
manager_df1.unionByName(Wrong_manager_df).show()

+---+-----+-----+-------+
| id| name|  sal|mngr_id|
+---+-----+-----+-------+
| 19|Sohan|50000|     18|
| 20| Sima|75000|     17|
| 19|Sohan|50000|     18|
| 20| Sima|75000|     17|
+---+-----+-----+-------+



In [0]:
wrong_column_data=[(19 ,50000, 18,'Sohan',10),
(20 ,75000,  17,'Sima',20)]
Wrong_schema =['id','sal','mngr_id','Name','bonus']
Wrong_manager_df=spark.createDataFrame(data=wrong_column_data,schema=Wrong_schema)

Below union will throw error as number of column is different in above data frame:

In [0]:
Wrong_manager_df.union(manager_df1).count()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4731901989430479>, line 1
----> 1 Wrong_manager_df.union(manager_df1).count()

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/sql/dataframe.py:4896, in DataFrame.union(self, other)
   4800 def union(self, other: "DataFrame") -> "DataFrame":
   4801     """Return a new :class:`DataFrame` containing the union of rows in this and another
   4802     :class:`DataFrame`.
   4803 
   (...)
   4894     +---+-----+
   4895     """
-> 4896     return DataFrame(self._jdf.unio

For this type of case we will do union by selecting the column:

In [0]:
Wrong_manager_df.select('id','sal','mngr_id','Name').union(manager_df1).show()

+---+-----+-------+-----+
| id|  sal|mngr_id| Name|
+---+-----+-------+-----+
| 19|50000|     18|Sohan|
| 20|75000|     17| Sima|
| 19|Sohan|  50000|   18|
| 20| Sima|  75000|   17|
+---+-----+-------+-----+



When column will not be in order in both dataframe then we can apply unionByName but if we have large dataframe and suppose for a column name, spelling is different then there we can't use unionByName :

In [0]:
wrong_column_data2=[(19 ,50000, 18,'Sohan'),
(20 ,75000,  17,'Sima')]
Wrong_schema2 =['id','sal','mngr_id','Nam']
Wrong_manager_df2=spark.createDataFrame(data=wrong_column_data2,schema=Wrong_schema2)

In [0]:
Wrong_manager_df.unionByName(Wrong_manager_df2).show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4731901989430486>, line 1
----> 1 Wrong_manager_df.unionByName(Wrong_manager_df2).show()

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/sql/dataframe.py:5010, in DataFrame.unionByName(self, other, allowMissingColumns)
   4932 def unionByName(self, other: "DataFrame", allowMissingColumns: bool = False) -> "DataFrame":
   4933     """Returns a new :class:`DataFrame` containing union of rows in this and another
   4934     :class:`DataFrame`.
   4935 
   (...)
   5008